In [15]:
import os
import sys

# 手动指定你 legato 项目的根目录路径
# 根据你之前提供的信息，路径应该是：
PROJECT_ROOT = r"C:/Users/20810/Desktop/Code/legato"

if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

import legato
print("Legato 模块加载成功！")

Legato 模块加载成功！


In [19]:
import torch
from pathlib import Path
from PIL import Image
from transformers import AutoProcessor, GenerationConfig
from legato.models import LegatoModel

snapshot_root = Path(PROJECT_ROOT) / "models" / "models--guangyangmusic--legato" / "snapshots"
snapshot_dirs = sorted([p for p in snapshot_root.iterdir() if p.is_dir()])
if not snapshot_dirs:
    raise FileNotFoundError(f"No local model snapshot found under: {snapshot_root}")

model_path = snapshot_dirs[0]
model = LegatoModel.from_pretrained(str(model_path), local_files_only=True)
print(model.config._name_or_path)

OSError: 页面文件太小，无法完成操作。 (os error 1455)

In [9]:
device = "cuda" if torch.cuda.is_available() else "cpu"
model = model.to(device).half() if device == "cuda" else model.to(device)
processor = AutoProcessor.from_pretrained(str(model_path), local_files_only=True)
print(f"Loaded local processor from: {model_path}")

Loaded local processor from: C:\Users\20810\Desktop\Code\legato\models\models--guangyangmusic--legato\snapshots\2d07c5d0e73186f2c0b12e35ea187bbc30dec18c


In [10]:
# Move to GPU
device = "cuda" if torch.cuda.is_available() else "cpu"
model = model.to(device)

In [11]:
# Load and process image
image = Image.open("./images/image2.jpg").convert("RGB")
inputs = processor(images=image, return_tensors="pt")
inputs = {k: v.to(device) for k, v in inputs.items()}

# Generate ABC notation
generation_config = GenerationConfig(
    max_length=2048,
    num_beams=10,
    repetition_penalty=1.1,
    pad_token_id=processor.tokenizer.pad_token_id,
    eos_token_id=processor.tokenizer.eos_token_id
)

with torch.no_grad():
    outputs = model.generate(**inputs, generation_config=generation_config)

# Decode output
abc_notation = processor.batch_decode(outputs, skip_special_tokens=True)[0]
print(abc_notation)

Truncation was not explicitly activated but `max_length` is provided a specific value, please use `truncation=True` to explicitly truncate examples to max length. Defaulting to 'longest_first' truncation strategy. If you encode pairs of sequences (GLUE-style) with the tokenizer you can select this strategy more precisely by providing a specific strategy to `truncation`.


X:1
%%score { 1 | 2 }
L:1/8
M:3/4
I:linebreak $
K:D
V:1 treble
V:2 bass
V:1
 z2 z2 A2 | A4 A2 | A4 ^G2 | [DG]4 [CG]2 |[M:4/4] [DF]6 [DA]2 | %5
[M:3/4] [Fd]4 d2 | d4 e2 | c6 |[M:1/4] [Ac]2 |[M:3/4] [Ad]4 [Ad]2 | %10
 c4 B2 |$ A4 ^G2 |[M:4/4] [CG]6 F2 |[M:3/4] F4 E2 | B4 C2 | %15
 D6 |] %16
V:2
 z2 z2 [F,A,D]2 | [F,A,D]4 [F,A,D]2 | [E,B,D]4 [E,B,D]2 | [A,,A,]4 [A,,A,]2 |[M:4/4] [D,A,]6 [F,A,]2 | %5
[M:3/4] [B,,B,]4[K:treble] [B,DF]2 | [B,D^G]4 [B,DG]2 | [A,EA]6 |[M:1/4] [G,E]2 |[M:3/4][K:bass] [F,D]4 [F,D]2 | %10
 [G,D]4 [G,D]2 |$ [F,A,D]4 [E,B,D]2 |[M:4/4] [A,,A,]6 [D,A,D]2 |[M:3/4] [G,B,]4 [G,B,]2 | [G,B,E]4 [A,,G,A,]2 | %15
 [D,F,A,]6 |] %16

